<!-- Forward pass (output): $$\sigma(A\sigma(AFW^{(1)})W^{(2)})$$
Loss: $$\mathcal{L}(\sigma(A\sigma(AFW^{(1)})W^{(2)}))$$
$\frac{\partial\mathcal{L}}{W^{(1)}}$:  -->

In [ ]:
import torch 
def message_function(A, F, W, b):
    return A @ F @ W + b
def ReLU(X):
    return X * (X > 0) 
def dReLU(X):
    return (X > 0).float()
def sigmoid(X):
    return 1 / (1 + torch.exp(-X))
def dsigmoid(X):
    return sigmoid(X) * (1 - sigmoid(X))
def cross_entropy_loss(y_pred, y_true):
    return -torch.mean(y_true * torch.log(y_pred + 1e-8) + (1 - y_true) * torch.log(1 - y_pred + 1e-8))
def dcross_entropy_loss(y_pred, y_true):
    return -(y_true / (y_pred + 1e-8) - (1 - y_true) / (1 - y_pred + 1e-8)) / y_pred.size(0)
def forward_pass(A, F, W1, W2, b1, b2):
    S1 = message_function(A, F, W1, b1)
    X1 = ReLU(S1)
    S2 = message_function(A, X1, W2, b2)
    X2 = sigmoid(S2) # Output layer with sigmoid activation
    return S1, X1, S2, X2
def backward_pass(A, F, W1, W2, b1, b2, S1, X1, S2, X2, target):
    dL_dX2 = dcross_entropy_loss(X2, target)
    dL_dS2 = dL_dX2 * dsigmoid(S2)
    dL_dW2 = X1.T @ (A.T @ dL_dS2)
    dL_db2 = torch.sum(dL_dS2, dim=0)
    dL_dX1 = A.T @ (dL_dS2 @ W2.T)
    dL_dS1 = dL_dX1 * dReLU(S1)
    dL_dW1 = F.T @ (A.T @ dL_dS1)
    dL_db1 = torch.sum(dL_dS1, dim=0)
    return dL_dW1, dL_db1, dL_dW2, dL_db2

torch.Size([5000, 784])
torch.Size([5000, 10])
